# Google Colab

In [1]:
# Get GPU info in Google Colab
GOOGLE_COLAB = False
if 'google.colab' in str(get_ipython()):
    GOOGLE_COLAB = True
    !pip install piexif
    !pip install datasets

# pip install piexif # for local runtime
# pip install datasets

# Imports

In [2]:
import kagglehub
import shutil
import os
import requests
from datasets import load_dataset

import cv2
import csv
import numpy as np
from PIL import Image
from tqdm import tqdm
from multiprocessing import Pool, cpu_count


# Notes

Please download the separate datasets and store them in the directory format below:

- `/chinese_landscapes_paintings_huggingface/*.jpg`
- `/chinese_landscapes_paintings/*.jpg`
- `/Landscape/*.jpg`

Post-processed images and metadata will be saved in directories formatted as `'post_processed_{folder}'`, stored separately.

To make the process faster and save space, it is recommended to connect Colab to a local runtime.


In [3]:
dataset_handle = "arnaud58/landscape-pictures" # Example handle
target_folder = "Landscape"

# download dataset using kagglehub
print("Downloading dataset...")
path = kagglehub.dataset_download(dataset_handle)

print(f"Source files downloaded to: {path}")

# 3. Move the files to your desired folder
os.makedirs(target_folder, exist_ok=True)

# Iterate over the files in the downloaded path and move them
for filename in os.listdir(path):
    source_file = os.path.join(path, filename)
    destination_file = os.path.join(target_folder, filename)
    
    # Use copy2 to preserve metadata, or move to save space
    shutil.copy2(source_file, destination_file) 

print(f"Files successfully copied to: {target_folder}")

Source files downloaded to: C:\Users\polski\.cache\kagglehub\datasets\arnaud58\landscape-pictures\versions\2
Files successfully copied to: Landscape


In [6]:
# chinese landscapes paintings using huggingface could be downloaded using the script below
ds = load_dataset("mingyy/chinese_landscape_paintings")
print(ds)
print(ds['train'][0])

# Define the directory to save the images
image_dir = "chinese_landscapes_paintings_huggingface"
os.makedirs(image_dir, exist_ok=True)
# Iterate through the dataset and save the images
limit = 35000 # 35k samples limit

for idx, example in enumerate(ds['train']):
    image = example['source']
    filename = example['filename']

    if idx >= limit:
        break
    # Save the image
    try:
        image_path = os.path.join(image_dir, filename)
        image.save(image_path)
        print(f"Downloaded {filename}, idx: {idx}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")

Resolving data files:   0%|          | 0/89 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/69 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['target', 'filename', 'image_caption', 'hed', 'source'],
        num_rows: 52564
    })
})
{'target': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x2C2A03EF360>, 'filename': '0000001.jpg', 'image_caption': 'a river running through a forest filled with rocks and trees', 'hed': <PIL.JpegImagePlugin.JpegImageFile image mode=L size=512x512 at 0x2C2A03EEFD0>, 'source': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1617x1617 at 0x2C2A03EFCE0>}
Downloaded 0000001.jpg, idx: 0
Downloaded 0000003.jpg, idx: 1
Downloaded 0000006.jpg, idx: 2
Downloaded 0000008.jpg, idx: 3
Downloaded 0000011.jpg, idx: 4
Downloaded 0000017.jpg, idx: 5
Downloaded 0000018.jpg, idx: 6
Downloaded 0000019.jpg, idx: 7
Downloaded 0000020.jpg, idx: 8
Downloaded 0000021.jpg, idx: 9
Downloaded 0000022.jpg, idx: 10
Downloaded 0000024.jpg, idx: 11
Downloaded 0000025.jpg, idx: 12
Downloaded 0000026.jpg, idx: 13
Downloaded 0000027.jpg, idx: 14
Dow

In [9]:
# chinese landscapes paintings using huggingface could be downloaded using the script below
ds = load_dataset("mingyy/chinese_landscape_paintings_1k")
print(ds)
print(ds['train'][0])

# Define the directory to save the images
image_dir = "chinese_landscapes_paintings_huggingface_1k"
os.makedirs(image_dir, exist_ok=True)
# Iterate through the dataset and save the images

for idx, example in enumerate(ds['train']):
    image = example['source']
    filename = example['filename']
    
    # Save the image
    try:
        image_path = os.path.join(image_dir, filename)
        image.save(image_path)
        print(f"Downloaded {filename}, idx: {idx}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")

DatasetDict({
    train: Dataset({
        features: ['target', 'filename', 'image_caption', 'source', 'hed'],
        num_rows: 1000
    })
})
{'target': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x2C2A03EFE10>, 'filename': 'EB4A4E9B-C687-453E-BC3D-C5D8A2BF4C10.jpg', 'image_caption': 'a view of a mountain with a cave in the middle', 'source': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1617x1617 at 0x2C2A05528B0>, 'hed': <PIL.JpegImagePlugin.JpegImageFile image mode=L size=512x512 at 0x2C2A05536F0>}
Downloaded EB4A4E9B-C687-453E-BC3D-C5D8A2BF4C10.jpg, idx: 0
Downloaded FF0368E7-BE8C-4414-80C6-E03B5D83F927.jpg, idx: 1
Downloaded IMG_2877.jpg, idx: 2
Downloaded IMG_2878.jpg, idx: 3
Downloaded IMG_2879.jpg, idx: 4
Downloaded IMG_2879_1.jpg, idx: 5
Downloaded IMG_2881.jpg, idx: 6
Downloaded IMG_2883.jpg, idx: 7
Downloaded IMG_2884.jpg, idx: 8
Downloaded IMG_3909.jpg, idx: 9
Downloaded IMG_3983.jpg, idx: 10
Downloaded IMG_4055.jpg, idx: 11
Downloaded IMG_

In [7]:
# Download another version of chinese paintings from huggingface
ds = load_dataset("WUYONGF/chinese_painting")
print(ds)
print(ds['train'][0])

# Define the directory to save the images
image_dir = "chinese_landscapes_paintings"
os.makedirs(image_dir, exist_ok=True)

# image idx to remove due to wrong domain (animal instead of landscape)
filter_idx = [0, 5, 22, 24, 25]

# Iterate through the dataset and save the images
for idx, example in enumerate(ds['train']):
    # print(example)
    image = example['image']
    filename = f"{idx}.jpg"

    if idx in filter_idx:
        print(f"Skipping index {idx} due to wrong domain.")
        continue
        
    # Save the image
    try:
        image_path = os.path.join(image_dir, filename)
        image.save(image_path)
        print(f"Downloaded {filename}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 30
    })
})
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=6336x9504 at 0x2C29FF66060>, 'text': 'a chinese painting of a monkey hanging from a tree branch'}
Skipping index 0 due to wrong domain.
Downloaded 1.jpg
Downloaded 2.jpg
Downloaded 3.jpg
Downloaded 4.jpg
Skipping index 5 due to wrong domain.
Downloaded 6.jpg
Downloaded 7.jpg
Downloaded 8.jpg
Downloaded 9.jpg
Downloaded 10.jpg
Downloaded 11.jpg
Downloaded 12.jpg
Downloaded 13.jpg
Downloaded 14.jpg
Downloaded 15.jpg
Downloaded 16.jpg
Downloaded 17.jpg
Downloaded 18.jpg
Downloaded 19.jpg
Downloaded 20.jpg
Downloaded 21.jpg
Skipping index 22 due to wrong domain.
Downloaded 23.jpg
Skipping index 24 due to wrong domain.
Skipping index 25 due to wrong domain.
Downloaded 26.jpg
Downloaded 27.jpg
Downloaded 28.jpg
Downloaded 29.jpg


In [10]:
RAW_FOLDERS = ["chinese_landscapes_paintings_huggingface_1k"]
IMG_SIZE = 512
NOISE_THRESHOLD = 0.2

# XDoG Filter Implementation
# def xdog(img, k=1.6, sigma=0.8, eps=-0.1, phi=50):
#     img = img.astype(np.float32) / 255.0
#     g1 = cv2.GaussianBlur(img, (0, 0), sigma)
#     g2 = cv2.GaussianBlur(img, (0, 0), sigma * k)
#     diff = g1 - g2
#     xdog_img = np.where(diff >= eps, 1.0, 1.0 + np.tanh(phi * diff))
#     return (xdog_img * 255).astype(np.uint8)
def xdog(img, sigma=0.5, k=1.2, eps=-0.05, phi=20):
    img = img.astype(np.float32) / 255.0
    # Gaussian blurs
    g1 = cv2.GaussianBlur(img, (0, 0), sigma)
    g2 = cv2.GaussianBlur(img, (0, 0), sigma * k)
    # Difference of Gaussians
    diff = g1 - g2
    # Soft thresholding instead of hard cutoff
    xdog = np.tanh(phi * diff)
    # Normalize to 0–255
    xdog = (xdog - xdog.min()) / (xdog.max() - xdog.min())
    xdog = 255 * xdog
    xdog = 255 - xdog
    return xdog.astype(np.uint8)

# Resize + center crop
def resize_512(img):
    h, w = img.shape[:2]
    side = min(h, w)
    y = (h - side) // 2
    x = (w - side) // 2
    img = img[y:y+side, x:x+side]
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LANCZOS4)

# Noise detector & thresholding
def is_noisy(sketch):
    black_ratio = (sketch < 128).mean()
    return black_ratio > NOISE_THRESHOLD

# Process each image
def process_image(args):
    cv2.setNumThreads(0) # Prevent OpenCV from oversubscribing threads
    file_path, file_id, out_root, source_name = args
    try:
        img = cv2.imread(file_path)
        if img is None:
            return None
        img_resized = resize_512(img)
        # Save original resized photo - to 512
        photo_path = f"photos/{file_id}.png"
        cv2.imwrite(os.path.join(out_root, photo_path), img_resized)
        # Normalized
        img_norm = (img_resized.astype(np.float32) / 255.0)
        img_norm_uint8 = (img_norm * 255).astype(np.uint8)
        photo_norm_path = f"photos_norm/{file_id}.png"
        cv2.imwrite(os.path.join(out_root, photo_norm_path), img_norm_uint8)
        # Canny - edge detection
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        canny = cv2.Canny(gray, 50, 150)
        canny_path = f"canny/{file_id}.png"
        cv2.imwrite(os.path.join(out_root, canny_path), canny)
        # XDoG - smooth edges
        xdog_sketch = xdog(gray)
        xdog_path = f"xdog/{file_id}.png"
        cv2.imwrite(os.path.join(out_root, xdog_path), xdog_sketch)
        # Cleaning the edges
        clean = xdog_sketch.copy()
        if is_noisy(clean):
            clean = cv2.medianBlur(clean, 5)
        clean_path = f"clean/{file_id}.png"
        cv2.imwrite(os.path.join(out_root, clean_path), clean)
        return {
            "id": file_id,
            "source": source_name,
            "photo": photo_path,
            "photo_norm": photo_norm_path,
            "canny": canny_path,
            "xdog": xdog_path,
            "clean": clean_path
        }
    except Exception as e:
        print("Error processing:", file_path, e)
        return None

# Build folder structure for metadata
def ensure_dirs(root):
    for sd in ["photos", "photos_norm", "canny", "xdog", "clean"]:
        os.makedirs(os.path.join(root, sd), exist_ok=True)

# Process one dataset folder
def process_single_dataset(raw_folder, multi_core=False):
    out_root = f"post_processed_{raw_folder}"
    ensure_dirs(out_root)
    all_files = []
    idx = 0
    # Collect images
    for root, _, files in os.walk(raw_folder):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
                file_id = f"{idx:06d}"
                full_path = os.path.join(root, f)
                all_files.append((full_path, file_id, out_root, raw_folder))
                idx += 1
    print(f"[{raw_folder}] Found {len(all_files)} images.")
    # Parallel processing
    results = []
    if multi_core:
        with Pool(cpu_count()) as p:
            for r in tqdm(p.imap_unordered(process_image, all_files), total=len(all_files)):
                if r is not None:
                    results.append(r)
    else:
        print("Running in DEBUG (Single Core) mode...")
        for args in tqdm(all_files):
            # Run directly without Pool to see errors
            r = process_image(args) 
            if r is not None:
                results.append(r)
    # Save metadata.csv
    csv_path = os.path.join(out_root, "metadata.csv")
    with open(csv_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["id", "source", "photo", "photo_norm", "canny", "xdog", "clean"])
        for row in results:
            writer.writerow([
                row["id"], row["source"],
                row["photo"], row["photo_norm"],
                row["canny"], row["xdog"], row["clean"]
            ])
    print(f"[OK] Saved metadata → {csv_path}")
    print(f"[DONE] Dataset processed: {raw_folder}\n")

# MAIN
if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    for folder in RAW_FOLDERS:
        process_single_dataset(folder, multi_core=False)


[chinese_landscapes_paintings_huggingface_1k] Found 1000 images.
Running in DEBUG (Single Core) mode...


100%|██████████| 1000/1000 [00:44<00:00, 22.43it/s]

[OK] Saved metadata → post_processed_chinese_landscapes_paintings_huggingface_1k\metadata.csv
[DONE] Dataset processed: chinese_landscapes_paintings_huggingface_1k

